In [2]:
import numpy as np
import torch
import torch.nn as nn


In [13]:
import pandas as pd
train = pd.read_csv("datasets/MELD/train_sent_emo.csv")
# print(train["Utterance"][0])
train

,Sr No.,Utterance,Speaker,Emotion,Sentiment,Dialogue_ID,Utterance_ID,Season,Episode,StartTime,EndTime
0,1,also I was the point person on my company’s tr...,Chandler,neutral,neutral,0,0,8,21,"00:16:16,059","00:16:21,731"
1,2,You must’ve had your hands full.,The Interviewer,neutral,neutral,0,1,8,21,"00:16:21,940","00:16:23,442"
2,3,That I did. That I did.,Chandler,neutral,neutral,0,2,8,21,"00:16:23,442","00:16:26,389"
3,4,So let’s talk a little bit about your duties.,The Interviewer,neutral,neutral,0,3,8,21,"00:16:26,820","00:16:29,572"
4,5,My duties? All right.,Chandler,surprise,positive,0,4,8,21,"00:16:34,452","00:16:40,917"
...,...,...,...,...,...,...,...,...,...,...,...
9984,10474,You or me?,Chandler,neutral,neutral,1038,13,2,3,"00:00:48,173","00:00:50,799"
9985,10475,"I got it. Uh, Joey, women don't have Adam's ap...",Ross,neutral,neutral,1038,14,2,3,"00:00:51,009","00:00:53,594"
9986,10476,"You guys are messing with me, right?",Joey,surprise,positive,1038,15,2,3,"00:01:00,518","00:01:03,520"
9987,10477,Yeah.,All,neutral,neutral,1038,16,2,3,"00:01:05,398","00:01:07,274"


In [ ]:
from torch.utils.data import Dataset, DataLoader
import os



class MELDDataset(Dataset):
    def __init__(self, data_path, mode_dir):
        df = pd.read_csv(data_path)
        self.data = []
        for i in range(len(df)):
            utterance = df["Utterance"][i]
            mp4_path = os.path.join(mode_dir, f"dia{df["Dialogue_ID"][i]}_utt{df["Utterance_ID"][i]}.mp4")

            emotion = df["Emotion"][i]
            to_append = {
                "text": utterance,
                "visual": mp4_path,
                "label": emotion
            }
            self.data.append(to_append)

    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, key):
        return self.data[key]

train_data_path = "datasets/MELD/train_sent_emo.csv"
dev_data_path = "datasets/MELD/dev_sent_emo.csv"
test_data_path = "datasets/MELD/test_sent_emo.csv"

train_dataset = MELDDataset(train_data_path, mode_dir="train")
dev_dataset = MELDDataset(dev_data_path, mode_dir="dev")
test_dataset = MELDDataset(test_data_path, mode_dir="test")


train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=16, shuffle=False)
test_data_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModel,
    VideoMAEImageProcessor,
    VideoMAEModel,
)

tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-small")

video_processor = VideoMAEImageProcessor.from_pretrained("MCG-NJU/videomae-base")

emotions_to_id = {
    "anger": 0,
    "disgust": 1,
    "fear": 2,
    "joy": 3,
    "neutral": 4,
    "sadness": 5,
    "surprise": 6
}


import cv2


def read_video(path, num_frames):
    capture = cv2.VideoCapture(path)
    try:
        frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))

        frames = []
        for index in np.linspace(0, frame_count - 1, num_frames):
            capture.set(cv2.CAP_PROP_POS_FRAMES, int(index))
            ok, frame = capture.read()
            frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        return frames
    finally:
        capture.release()


class MultimodalModel(nn.Module):
    def __init__(self, text_model, visual_model):
        super().__init__()
        self.text_model = text_model
        self.visual_model = visual_model
                # Freeze pretrained encoders
        for param in self.text_model.parameters():
            param.requires_grad = False

        for param in self.visual_model.parameters():
            param.requires_grad = False
        
        self.classfier = nn.Sequential(
            nn.Linear(768+768, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 7)
        )


    def train(self, mode=True):
        super().train(mode)
        # Frozen encoders should remain deterministic; classifier dropout still trains.
        self.text_model.eval()
        self.visual_model.eval()
        return self

    def forward(self, text, visual):
        device = next(self.classfier.parameters()).device
        text_tokens = tokenizer(
            text, padding=True, truncation=True, max_length=512, return_tensors="pt"
        ).to(device)
        frames = [read_video(path, self.visual_model.config.num_frames) for path in visual]
        visual_tokens = video_processor(frames, return_tensors="pt").to(device)
        with torch.no_grad():
            text_hidden = self.text_model(**text_tokens).last_hidden_state
            mask = text_tokens["attention_mask"].unsqueeze(-1).to(text_hidden.dtype)
            text = (text_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            visual = self.visual_model(**visual_tokens).last_hidden_state.mean(dim=1)

        combined = torch.cat([text, visual], dim=-1)
        logits = self.classfier(combined)
        return logits


In [ ]:
# Training loop
text_model = AutoModel.from_pretrained(
    "microsoft/deberta-v3-small"
)



visual_model = VideoMAEModel.from_pretrained(
    "MCG-NJU/videomae-base"
)

model = MultimodalModel(
    text_model=text_model,
    visual_model=visual_model
)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device)
print("Device:", device)

num_epochs = 10
learning_rate = 2e-3

optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

def batch_labels(batch):
    return torch.tensor([emotions_to_id[label] for label in batch["label"]], dtype=torch.long, device=device)


@torch.no_grad()
def accuracy(loader):
    model.eval()
    correct, total = 0, 0
    for batch in loader:
        # converst datasets_label to numbers
        labels = batch_labels(batch)
        # got preprocessed in the model
        predictions = model(batch["text"], batch["visual"]).argmax(dim=-1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
    return correct / total



for epoch in range(num_epochs):
    model.train()
    total_loss, total_examples = 0.0, 0
    for batch in train_loader:
        optimizer.zero_grad()
        logits = model(batch["text"], batch["visual"])
        # converts dataset labels to numbers
        labels = batch_labels(batch)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        total_examples += labels.size(0)

    print(f"Epoch {epoch + 1}/{num_epochs} | loss: {total_loss / total_examples:.4f} | "
          f"validation accuracy: {accuracy(dev_loader):.2%}")

print(f"Test accuracy: {accuracy(test_data_loader):.2%}")
# Only the classifier learns; reuse the same pretrained encoders when loading it.
torch.save({"classifier": model.classfier.state_dict(), "labels": emotions_to_id}, "meld_classifier.pt")
